In [ ]:
!git clone https://github.com/hasmiig/Machine_Learning_Project
%cd Machine_Learning_Project


In [1]:
import sys
import os

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


## Load data



In [2]:
from google.colab import drive
drive.mount('/content/drive')
shared_folder = "Machine_learning_project/Data/Raw"

target_df = pd.read_csv(f"/content/drive/MyDrive/{shared_folder}/target.csv")
feature_df = pd.read_csv(f"/content//drive/MyDrive/Machine_learning_project/Data/Processed/listings_image_availability.csv")
test_ids = np.load(f"/content/drive/MyDrive/{shared_folder}/test_ids.npy")
train_ids = np.load(f"/content/drive/MyDrive/{shared_folder}/train_ids.npy")


Mounted at /content/drive


In [ ]:
feature_df.head()


In [3]:
from pathlib import Path

import requests

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torch


In [4]:
FEATURE_COLUMN   = "picture_url"   # column containing the image URLs
LABEL_COLUMN = "price"       # column used as the label


## Build train / test sets

Non-available images carry no information (they can't be downloaded/used), so we
filter them out here instead of trying to estimate or impute them.


In [5]:
#Creating the train and test datasets
available_ids = feature_df.loc[feature_df['available'], 'id']

# keep the "id" column too - we need it below to cache images to disk by id
X_train = feature_df[feature_df['id'].isin(train_ids) & feature_df['id'].isin(available_ids)][['id', 'picture_url']]
y_train = target_df[target_df['id'].isin(train_ids) & target_df['id'].isin(available_ids)][['log_price']]

x_test = feature_df[feature_df['id'].isin(test_ids) & feature_df['id'].isin(available_ids)][['id', 'picture_url']]
y_test = target_df[target_df['id'].isin(test_ids) & target_df['id'].isin(available_ids)][['log_price']]


In [6]:
total_train = feature_df['id'].isin(train_ids).sum()
total_test = feature_df['id'].isin(test_ids).sum()

available_train = len(X_train)
available_test = len(x_test)

print(f"Train: {available_train} / {total_train} available ({available_train/total_train:.1%})")
print(f"Test:  {available_test} / {total_test} available ({available_test/total_test:.1%})")


Train: 4614 / 4926 available (93.7%)
Test:  1157 / 1232 available (93.9%)


In [7]:
import os
import time
from io import BytesIO
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import requests
from PIL import Image
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from torchvision import models


In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


Using device: cpu


### One-time parallel image download / cache

Downloads every image we'll actually use (train + test) once, using a thread pool
(HTTP requests are I/O bound, so threads give a big speedup here). Already-cached
files are skipped.


In [9]:
CACHE_DIR = Path("/content/image_cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

IMG_TIMEOUT = 10
MAX_DOWNLOAD_WORKERS = 32  # threads for concurrent downloads (I/O bound, not CPU bound)

def _cache_path(img_id) -> Path:
    return CACHE_DIR / f"{img_id}.jpg"

IMG_SIZE = 224  # resize once at download time so later steps never resize again

def download_one(img_id, url) -> None:
    path = _cache_path(img_id)
    if path.exists():
        # fix up any stale full-resolution files left over from earlier runs
        try:
            existing = Image.open(path)
            if existing.size == (IMG_SIZE, IMG_SIZE):
                return  # already cached and already the right size
            existing.convert("RGB").resize((IMG_SIZE, IMG_SIZE)).save(path, "JPEG", quality=90)
            return
        except Exception:
            pass  # corrupt cached file - fall through and re-download
    try:
        resp = requests.get(url, timeout=IMG_TIMEOUT)
        resp.raise_for_status()
        img = Image.open(BytesIO(resp.content)).convert("RGB")
    except Exception:
        # shouldn't happen since we already filtered to "available" ids,
        # but fall back to a placeholder just in case a URL goes stale
        img = Image.new("RGB", (IMG_SIZE, IMG_SIZE))
    img = img.resize((IMG_SIZE, IMG_SIZE))  # small + no resizing needed later
    img.save(path, "JPEG", quality=90)

def download_all(df: pd.DataFrame, id_col: str = "id", url_col: str = "picture_url",
                  max_workers: int = MAX_DOWNLOAD_WORKERS) -> None:
    tasks = list(zip(df[id_col], df[url_col]))
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [executor.submit(download_one, img_id, url) for img_id, url in tasks]
        for _ in tqdm(as_completed(futures), total=len(futures), desc="Caching images"):
            pass


In [10]:
start = time.time()
all_images_df = pd.concat([X_train[["id", "picture_url"]], x_test[["id", "picture_url"]]]).drop_duplicates("id")
download_all(all_images_df)
print(f"Cached {len(all_images_df)} images in {time.time() - start:.1f}s")


Caching images:   0%|          | 0/5771 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (96000000 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


Cached 5771 images in 1177.1s


### Load every cached image into memory once


In [11]:
def load_to_memory(ids) -> dict:
    cache = {}
    for img_id in tqdm(ids, desc="Loading images into RAM"):
        path = _cache_path(img_id)
        try:
            img = Image.open(path).convert("RGB")
            # Resize - guards against stale/full-resolution cache files from earlier runs blowing up RAM
            if img.size != (IMG_SIZE, IMG_SIZE):
                img = img.resize((IMG_SIZE, IMG_SIZE))
        except Exception:
            img = Image.new("RGB", (IMG_SIZE, IMG_SIZE))
        # store as uint8 tensor (C, H, W) - no normalisation yet, keeps memory low
        cache[img_id] = torch.from_numpy(np.array(img)).permute(2, 0, 1).contiguous()
        del img
    return cache

image_memory_cache = load_to_memory(all_images_df["id"].tolist())

approx_gb = sum(t.numel() for t in image_memory_cache.values()) / 1e9
print(f"In-memory image cache: {len(image_memory_cache)} images, ~{approx_gb:.2f} GB (uint8)")


Loading images into RAM:   0%|          | 0/5771 [00:00<?, ?it/s]

In-memory image cache: 5771 images, ~0.87 GB (uint8)


## Dataset: read from the in-memory tensor cache


In [12]:
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
IMAGENET_STD  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

class UrlImageDataset(Dataset):
    def __init__(self, X: pd.DataFrame, Y: pd.DataFrame, image_cache: dict):
        self.ids    = X["id"].tolist()
        self.prices = Y.iloc[:, 0].tolist()
        self.image_cache = image_cache

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        img_id = self.ids[idx]
        img = self.image_cache.get(img_id)
        if img is None:
            img = torch.zeros(3, IMG_SIZE, IMG_SIZE, dtype=torch.uint8)

        # uint8 [0,255] -> float [0,1] -> ImageNet-normalised, done cheaply here
        img = img.float().div_(255.0)
        img = (img - IMAGENET_MEAN) / IMAGENET_STD

        return img, torch.tensor(self.prices[idx], dtype=torch.float32)


In [13]:
class ResNet50Regressor(nn.Module):
    def __init__(self, pretrained: bool = True, freeze_backbone: bool = True):
        super().__init__()
        weights = models.ResNet50_Weights.DEFAULT if pretrained else None
        backbone = models.resnet50(weights=weights)

        # strip the original classification head
        in_features = backbone.fc.in_features          # 2048 for ResNet-50
        backbone.fc = nn.Identity()                    # strip the classifier
        self.backbone = backbone

        if freeze_backbone:
            # Freeze everything except the last residual block (layer4).
            # Cuts the backward-pass cost a lot while still letting the model
            # adapt its highest-level features to this task.
            for name, param in self.backbone.named_parameters():
                if not name.startswith("layer4"):
                    param.requires_grad = False

        # Regression head: 2048 -> 256 -> 1
        self.head = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        features = self.backbone(x)            # (B, 2048)
        return self.head(features).squeeze(1)  # (B,)


In [14]:
train_dataset = UrlImageDataset(X_train, y_train, image_memory_cache)
val_dataset   = UrlImageDataset(x_test,  y_test,  image_memory_cache)

# Data now entirely in RAM
NUM_WORKERS = 2

train_loader = DataLoader(
    train_dataset, batch_size=32, shuffle=True,
    pin_memory=(device.type == "cuda"), num_workers=NUM_WORKERS,
    persistent_workers=(NUM_WORKERS > 0),
)
val_loader = DataLoader(
    val_dataset, batch_size=32, shuffle=False,
    pin_memory=(device.type == "cuda"), num_workers=NUM_WORKERS,
    persistent_workers=(NUM_WORKERS > 0),
)

model   = ResNet50Regressor(pretrained=True, freeze_backbone=True).to(device)
opt     = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)
loss_fn = nn.MSELoss()
scaler  = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))


Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 116MB/s]
/tmp/ipykernel_1322/3152625260.py:23: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler  = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))


In [ ]:
epochs = 5

for epoch in range(epochs):
    epoch_start = time.time()

    # -- Train --------------------------------------------------------------
    model.train()
    train_loss = 0.0

    for imgs, prices in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
        imgs   = imgs.to(device, non_blocking=True)
        prices = prices.to(device, non_blocking=True)

        opt.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
            pred = model(imgs)
            loss = loss_fn(pred, prices)

        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()

        train_loss += loss.item()

    train_rmse = (train_loss / len(train_loader)) ** 0.5

    print(f"Epoch {epoch+1}/{epochs}  "
          f"Train RMSE={train_rmse:.4f}  "
          f"({time.time() - epoch_start:.1f}s)")


Epoch 1/5:   0%|          | 0/145 [00:00<?, ?it/s]

/tmp/ipykernel_1322/4263650812.py:15: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):


Epoch 1/5  Train RMSE=1.5953  (1668.5s)


Epoch 2/5:   0%|          | 0/145 [00:00<?, ?it/s]

/tmp/ipykernel_1322/4263650812.py:15: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):


Epoch 2/5  Train RMSE=0.5958  (1691.2s)


Epoch 3/5:   0%|          | 0/145 [00:00<?, ?it/s]

In [ ]:
#Saving a checkpoint of the model
torch.save({
    "model_state_dict": model.state_dict(),
    "epoch": epoch,
    "loss": loss.item()
}, "checkpoint.pth")

Now we evaluate the model on the validation data set:

In [ ]:
predictions = []
targets = []

with torch.no_grad():
    for imgs, prices in val_loader:
        imgs = imgs.to(device)
        preds = model(imgs).cpu().numpy()

        predictions.extend(preds)
        targets.extend(prices.numpy())

predictions = np.array(predictions)
targets = np.array(targets)

rmse = np.sqrt(mean_squared_error(targets, predictions))
mae = mean_absolute_error(targets, predictions)
r2 = r2_score(targets, predictions)

print(f"RMSE : {rmse:,.2f}")
print(f"MAE  : {mae:,.2f}")
print(f"R²   : {r2:.3f}")

In [ ]:
#Looking at individual predictions
results = pd.DataFrame({
    "Actual": targets,
    "Predicted": predictions,
})

results["Error"] = results["Predicted"] - results["Actual"]
results["Abs Error"] = results["Error"].abs()

print(results.head(20))

In [ ]:
#Predicted vs actual
plt.figure(figsize=(6,6))
plt.scatter(targets, predictions, alpha=0.5)

mn = min(targets.min(), predictions.min())
mx = max(targets.max(), predictions.max())
plt.plot([mn, mx], [mn, mx], "r--")

plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.title("Predicted vs Actual")
plt.show()

In [ ]:
#Comparison against baseline
baseline_pred = np.full_like(targets, targets.mean())

baseline_rmse = np.sqrt(mean_squared_error(targets, baseline_pred))
baseline_mae = mean_absolute_error(targets, baseline_pred)

print("\nBaseline (predict mean price)")
print(f"RMSE: {baseline_rmse:,.2f}")
print(f"MAE : {baseline_mae:,.2f}")

print("\nModel improvement")
print(f"RMSE improvement: {(baseline_rmse - rmse):,.2f}")
print(f"MAE improvement : {(baseline_mae - mae):,.2f}")